Búsqueda, descarga preparación de encuestas de Los Mexicanos

In [7]:
# ruta encuestas
ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re
import time

base_url = "http://www.losmexicanos.unam.mx/"
domain = urlparse(base_url).netloc
pattern = re.compile(r'\.(pdf|dat|sav)$', re.IGNORECASE)

visited_pages = set()
found_files = set()

def crawl(url, delay=0.5):
    if url in visited_pages:
        return
    visited_pages.add(url)
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print(f"Error accessing {url}: {e}")
        return

    # Only parse HTML pages
    content_type = response.headers.get("Content-Type", "")
    if "text/html" not in content_type:
        return

    soup = BeautifulSoup(response.text, "html.parser")
    
    # Check all links on the page
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        full_url = urljoin(url, href)
        parsed_href = urlparse(full_url)

        # Skip if not on same domain
        if parsed_href.netloc != domain:
            continue

        # If link is a file link, add it
        if pattern.search(full_url):
            found_files.add(full_url)
        else:
            # Otherwise crawl the page if not already visited
            # You could restrict file extensions to only common page types (e.g., .html, .php) if needed.
            if full_url not in visited_pages:
                crawl(full_url, delay)
    
    # Optionally delay between requests so as not to overwhelm the server
    time.sleep(delay)

crawl(base_url)



print("Found file URLs:")
for file_url in found_files:
    print(file_url)

found_files = list(found_files)

Error accessing http://www.losmexicanos.unam.mx/juegosazar/prologo.html: 404 Client Error: Not Found for url: http://www.losmexicanos.unam.mx/juegosazar/prologo.html
Error accessing http://www.losmexicanos.unam.mx/nota_preliminar.html: 404 Client Error: Not Found for url: http://www.losmexicanos.unam.mx/nota_preliminar.html
Error accessing http://www.losmexicanos.unam.mx/presentacion.html: 404 Client Error: Not Found for url: http://www.losmexicanos.unam.mx/presentacion.html
Error accessing http://www.losmexicanos.unam.mx/perfil_encuestados.html: 404 Client Error: Not Found for url: http://www.losmexicanos.unam.mx/perfil_encuestados.html
Error accessing http://www.losmexicanos.unam.mx/autores.html: 404 Client Error: Not Found for url: http://www.losmexicanos.unam.mx/autores.html
Error accessing http://www.losmexicanos.unam.mx/indice.html: 404 Client Error: Not Found for url: http://www.losmexicanos.unam.mx/indice.html
Error accessing http://www.losmexicanos.unam.mx/encuesta_nacional.ht

In [9]:
import pyreadstat
import os  # newly added to handle file paths

noms_bases = [nom for nom in found_files if nom.endswith('.sav') and 'Movilidad' not in nom and 'linea' not in nom]
  
enc_dict = {}
for link in noms_bases:
    filename = link.split("/")[-1]
    print(f"Downloading and processing: {filename}")
    
    # Download the file
    response = requests.get(link)
    response.raise_for_status()
    
    # Save the file locally
    with open(os.path.join(ruta_enc, filename), "wb") as file_out:  # modified to save in ruta_enc
        file_out.write(response.content)
    
    # Read the .sav file
    df, meta = pyreadstat.read_sav(os.path.join(ruta_enc, filename))
    
    filename = filename.replace('-', '_').replace(' ', '_')

    prefix_lst = ["Tercera_Encuesta_Nacional_de_" , "Encuesta_Nacional_sobre_" , "Encuesta_Nacional_de_" , "encuesta-nacional-", "encuesta_nacional_en_vivienda_"]

    prefix = ""
    for p in prefix_lst:
        if filename.startswith(p):
            prefix = p
            break

    if filename.startswith(prefix):
        trimmed = filename[len(prefix):]
    else:
        trimmed = filename
    new_base = "_".join(trimmed.split()[:3])
    filename = f"{new_base}"

    meta = {
        "variable_value_labels": meta.variable_value_labels,
        "column_names_to_labels": meta.column_names_to_labels
    }

    # Store the dataframe and metadata in the dictionary
    enc_dict[filename] = {"dataframe": df, "metadata": meta}

enc_dict = {key.replace('.sav', ''): value for key, value in enc_dict.items()}



### procesamiento de cuestionarios y metadatos

In [ ]:
# ajuste de nombres para llaves
def replace_latin_characters(text):
    """
    Replaces Latin American characters with their ASCII equivalents.
    """
    replacements = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
        'ñ': 'n', 'Ñ': 'N'
    }
    for latin_char, ascii_char in replacements.items():
        text = text.replace(latin_char, ascii_char)
    return text


In [ ]:
# # importar encuestas en pickle, enc_dict tiene los resultados

# import pickle

# ruta_enc= '/Users/salvadorVMA/Google Drive/01 Proyectos/2025/navegador/encuestas'

# with open(f'{ruta_enc}/encs.pkl', 'rb') as f:
#     enc_dict = pickle.load(f)
#     print('Pickle file loaded successfully')

# enc_dict = {replace_latin_characters(k.upper()) : v for k, v in enc_dict.items()}

Pickle file loaded successfully


In [ ]:
# enc_nom_dict tiene los nombres de las encuestas

enc_nom_dict = {
    "IDE": "Identidad y Valores",
    "MED": "Medio Ambiente",
    "POB": "Pobreza",
    "CUL": "Cultura Política",
    "REL": "Religión, Secularización y Laicidad",
    "SEG": "Seguridad Pública",
    "SAL": "Salud",
    "IND": "Indígenas",
    "SOC": "Sociedad de la Información",
    "ENV": "Envejecimiento",
    "DER": "Derechos Humanos, Discriminación y Grupos Vulnerables",
    "COR": "Corrupción y Cultura de la Legalidad",
    "HAB": "La Condición de Habitabilidad de Vivienda en México",
    "GLO": "Globalización",
    "JUS": "Justicia",
    "JUE": "Juegos de Azar",
    "MIG": "Migración",
    "FED": "Federalismo",
    "GEN": "Género",
    "CON": "Cultura Constitucional",
    "DEP": "Cultura, Lectura y Deporte",
    "ECO": "Economía y Empleo",
    "NIN": "Niños, Adolescentes y Jóvenes",
    "FAM": "Familia",
    "CIE": "Ciencia y Tecnología",
    "EDU": "Educación"
}

# Apply the function to the items of enc_nom_dict
enc_nom_dict = {k: replace_latin_characters(v.upper()) for k, v in enc_nom_dict.items()}
enc_nom_dict = {k: v.replace(" ", "_").replace(',', '') for k, v in enc_nom_dict.items()}
enc_nom_dict = {v: k for k, v in enc_nom_dict.items()}
enc_nom_dict


{'IDENTIDAD_Y_VALORES': 'IDE',
 'MEDIO_AMBIENTE': 'MED',
 'POBREZA': 'POB',
 'CULTURA_POLITICA': 'CUL',
 'RELIGION_SECULARIZACION_Y_LAICIDAD': 'REL',
 'SEGURIDAD_PUBLICA': 'SEG',
 'SALUD': 'SAL',
 'INDIGENAS': 'IND',
 'SOCIEDAD_DE_LA_INFORMACION': 'SOC',
 'ENVEJECIMIENTO': 'ENV',
 'DERECHOS_HUMANOS_DISCRIMINACION_Y_GRUPOS_VULNERABLES': 'DER',
 'CORRUPCION_Y_CULTURA_DE_LA_LEGALIDAD': 'COR',
 'LA_CONDICION_DE_HABITABILIDAD_DE_VIVIENDA_EN_MEXICO': 'HAB',
 'GLOBALIZACION': 'GLO',
 'JUSTICIA': 'JUS',
 'JUEGOS_DE_AZAR': 'JUE',
 'MIGRACION': 'MIG',
 'FEDERALISMO': 'FED',
 'GENERO': 'GEN',
 'CULTURA_CONSTITUCIONAL': 'CON',
 'CULTURA_LECTURA_Y_DEPORTE': 'DEP',
 'ECONOMIA_Y_EMPLEO': 'ECO',
 'NINOS_ADOLESCENTES_Y_JOVENES': 'NIN',
 'FAMILIA': 'FAM',
 'CIENCIA_Y_TECNOLOGIA': 'CIE',
 'EDUCACION': 'EDU'}

In [ ]:
import re

# pregs_dict contiene SOLO las preguntas de la encuesta (NO tiene ponderadores)

#enc_ID = 'CIENCIA_Y_TECNOLOGIA'

# inicio de loop

pregs_agg_dict = {}

for ky in enc_dict.keys():

    tmp_pregs_dict = enc_dict[ky]['metadata']['column_names_to_labels']
    tmp_pregs_dict = {k: v for k, v in tmp_pregs_dict.items() if k.startswith('p') or k.startswith('sd')}

    # limpiar nombres de preguntas
    rgx_st = r'^\s*\d+\.*\s'

    for k, v in tmp_pregs_dict.items():
        if isinstance(v, str) and re.match(rgx_st, v):
            tmp_pregs_dict[k] = re.sub(rgx_st, '', v).strip()

    # agregar enc_ID a preguntas y textos
    tmp_pregs_dict = {'|'.join([k, enc_nom_dict[ky]]) : '|'.join([ky, v]) for k, v in tmp_pregs_dict.items()}
    pregs_agg_dict.update(tmp_pregs_dict)



# Split pregs_dict into two dictionaries
ses_dict = {k: v for k, v in pregs_agg_dict.items() if k.startswith('sd')}
pregs_dict = {k: v for k, v in pregs_agg_dict.items() if k.startswith('p')}

# eliminar preguntas redundantes
pregs_dict = {k: v for k, v in pregs_dict.items() if not ('a' in k or 'a_1' in k or '2°' in v or '3°' in v or '2 menc' in v or '3 menc' in v or 'ot' in k)}



list(pregs_dict.values()), list(ses_dict.values())

(['IDENTIDAD_Y_VALORES|Con la palabra maíz, yo asocio comida, mercado, animales. Dígame por favor, tres palabras que asocies con la palabra MÉXICO. 1° MENCIÓN',
  'IDENTIDAD_Y_VALORES|Y ahora, voy a pedir que me diga, por favor, tres palabras que asocie con la palabra MEXICANO. 1° MENCIÓN',
  'IDENTIDAD_Y_VALORES|De los siguientes lugares que le voy a mencionar, dígame qué tan unido se siente a su barrio o colonia',
  'IDENTIDAD_Y_VALORES|De los siguientes lugares que le voy a mencionar, dígame qué tan unido se siente a su localidad o pueblo',
  'IDENTIDAD_Y_VALORES|De los siguientes lugares que le voy a mencionar, dígame qué tan unido se siente a su estado',
  'IDENTIDAD_Y_VALORES|De los siguientes lugares que le voy a mencionar, dígame qué tan unido se siente a su región',
  'IDENTIDAD_Y_VALORES|De los siguientes lugares que le voy a mencionar, dígame qué tan unido se siente a México',
  'IDENTIDAD_Y_VALORES|De los siguientes lugares que le voy a mencionar, dígame qué tan unido se si

In [ ]:
# a pickle
import pickle 

readme_dict = {
    'enc_dict': {
        'df': 'dict con proyecto: df',
        'metadata': {
            'variable_value_labels': {'pxx': {'d.0: lbl'}},
            'column_names_to_labels': {'pxx': 'preg'}
            },
    'enc_nom_dict': 'dict con nombres proyecto: TPR',
    'pregs_dict': 'dict con preg|TPR: pregunta',
    'ses_dict': 'dict con sd|TPR: pregunta SES'
    }}

los_mex_dict = {
    'enc_dict': enc_dict,
    'enc_nom_dict': enc_nom_dict,
    'pregs_dict': pregs_dict,
    'ses_dict': ses_dict,
    'readme_dict': readme_dict
}


with open(f'{ruta_enc}/los_mex_dict.pkl', 'wb') as f:
    pickle.dump(los_mex_dict, f)
    print('Pickle file saved successfully')

In [10]:
# enc_dict a pickle
import pickle
import os  # newly added to handle file paths

with open('encs.pkl', 'wb') as f:
    pickle.dump(enc_dict, open(os.path.join(ruta_enc, 'encs.pkl'), 'wb'))

In [ ]:




# Get the first URL in found_files that ends with .sav
sav_url = next(url for url in found_files if url.lower().endswith('.sav'))
print("Downloading:", sav_url)

# Download the file
response = requests.get(sav_url)
response.raise_for_status()  # Ensure the request was successful

# Extract the filename from the URL
sav_filename = sav_url.split("/")[-1]

# Save the file locally
with open(os.path.join(ruta_enc, sav_filename), "wb") as file_out:  # modified to save in ruta_enc
    file_out.write(response.content)
print("Downloaded", sav_filename)

# Examine the SPSS file metadata using pyreadstat

# read_sav returns the DataFrame and the metadata
df, meta = pyreadstat.read_sav(sav_filename)

# Display some metadata information
print("Metadata column names:")
print(meta.column_names)
print("\nVariable labels:")
print(meta.variable_value_labels)



Downloading: http://www.losmexicanos.unam.mx/MexicanosConstitucion/encuesta_nacional/base_datos/Tercera_Encuesta_Nacional_de_Cultura_Constitucional.sav
Downloaded Tercera_Encuesta_Nacional_de_Cultura_Constitucional.sav
Metadata column names:
['ID_BASE', 'EDO', 'MUNI', 'LOC', 'AGEB', 'FOLIO', 'DIA_1', 'MES_1', 'HR_INI_1_1', 'HR_FIN_1_1', 'DURA_1', 'RESUL_1', 'DIA_2', 'MES_2', 'HR_INI_2_1', 'HR_FIN_2_1', 'DURA_2', 'RESUL_2', 'DIA_3', 'MES_3', 'HR_INI_3_1', 'HR_FIN_3_1', 'DURA_3', 'RESUL_3', 'DIA_4', 'MES_4', 'HR_INI_4_1', 'HR_FIN_4_1', 'DURA_4', 'RESUL_4', 'DIA_5', 'MES_5', 'DIA_6', 'MES_6', 'DIA_7', 'MES_7', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7_1', 'P7_2', 'P7_3', 'P8_1', 'P8_2', 'P8_3', 'P8_4', 'P9_1', 'P9_1a_1', 'P9_2', 'P9_2a_1', 'P9_3', 'P9_3a_1', 'P10', 'P10a_1', 'P11', 'P11a_1', 'P12_1', 'P12_2', 'P12_3', 'P12_4', 'P12_5', 'P13_1', 'P13_2', 'P13_3', 'P13_4', 'P13_5', 'P13_6', 'P13_7', 'P13_8', 'P13_9', 'P13_10', 'P13_11', 'P13_12', 'P13_13', 'P13_14', 'P13_15', 'P13_16', 'P13_1

dict_keys(['column_names', 'column_labels', 'column_names_to_labels', 'file_encoding', 'number_columns', 'number_rows', 'variable_value_labels', 'value_labels', 'variable_to_label', 'notes', 'original_variable_types', 'readstat_variable_types', 'table_name', 'missing_ranges', 'missing_user_values', 'variable_storage_width', 'variable_display_width', 'variable_alignment', 'variable_measure', 'creation_time', 'modification_time', 'file_label', 'file_format'])

In [23]:
meta.column_names_to_labels

{'ID_BASE': None,
 'EDO': 'Estado',
 'MUNI': 'Municipio',
 'LOC': 'Localidad',
 'AGEB': 'Ageb',
 'FOLIO': 'Folio',
 'DIA_1': 'Dia',
 'MES_1': 'Mes',
 'HR_INI_1_1': 'Hora de Inicio',
 'HR_FIN_1_1': 'Hora de Fin',
 'DURA_1': 'Dura',
 'RESUL_1': 'Resultado',
 'DIA_2': 'Dia',
 'MES_2': 'Mes',
 'HR_INI_2_1': 'Hora de Inicio',
 'HR_FIN_2_1': 'Hora de Fin',
 'DURA_2': 'Dura',
 'RESUL_2': 'Resultado',
 'DIA_3': 'Dia',
 'MES_3': 'Mes',
 'HR_INI_3_1': 'Hora de Inicio',
 'HR_FIN_3_1': 'Hora de Fin',
 'DURA_3': 'Dura',
 'RESUL_3': 'Resultado',
 'DIA_4': 'Dia',
 'MES_4': 'Mes',
 'HR_INI_4_1': 'Hora de Inicio',
 'HR_FIN_4_1': 'Hora de Fin',
 'DURA_4': 'Dura',
 'RESUL_4': 'Resultado',
 'DIA_5': 'Dia',
 'MES_5': 'Mes',
 'DIA_6': 'Dia',
 'MES_6': 'Mes',
 'DIA_7': 'Dia',
 'MES_7': 'Mes',
 'P1': 'P1. Comparada con la situación que tenía el país hace un año, ¿cómo diría usted que es la situación actual del país: mejor o peor?',
 'P2': 'P2. De las siguientes palabras, ¿con cuál está usted más de acuerdo pa

In [22]:
meta.variable_value_labels

{'P1': {1.0: ' Mejor',
  2.0: ' Peor',
  3.0: ' Igual de bien',
  4.0: ' Igual de mal',
  8.0: ' NS',
  9.0: ' NC'},
 'P2': {1.0: ' Prometedora',
  2.0: ' Preocupante',
  3.0: ' Tranquila',
  4.0: ' Peligrosa',
  5.0: ' Con oportunidades',
  6.0: ' Mejor que antes',
  7.0: ' Más o menos',
  8.0: ' Peor que antes',
  9.0: ' Otra',
  98.0: ' NS',
  99.0: ' NC'},
 'P3': {1.0: ' Va a mejorar',
  2.0: ' Va a empeorar',
  3.0: ' Va a seguir igual de bien',
  4.0: ' Va a seguir igual de mal',
  8.0: ' NS',
  9.0: ' NC'},
 'P4': {1.0: ' Mucho',
  2.0: ' Algo',
  3.0: ' Poco',
  4.0: ' Nada',
  8.0: ' NS',
  9.0: ' NC'},
 'P5': {1.0: ' Siempre',
  2.0: ' Nunca',
  3.0: ' A veces',
  4.0: ' Sólo en ocasiones muy especiales',
  8.0: ' NS',
  9.0: ' NC'},
 'P6': {1.0: ' Televisión',
  2.0: ' Radio',
  3.0: ' Internet',
  4.0: ' Periódicos',
  5.0: ' Revistas',
  6.0: ' Amigos y conocidos',
  7.0: ' Otro',
  8.0: ' NS',
  9.0: ' NC'},
 'P7_1': {1.0: ' Acuerdo',
  2.0: ' Desacuerdo',
  3.0: ' Acuerd